# ODI to Databricks Migration: `TRG_EMP.txt`

**Conversion Timestamp:** `2024-07-30 12:00:00`

This notebook contains the converted Spark SQL logic for loading data into the `TRG_EMP` table from the `EMPLOYEES` source.
The original ODI process included simple `INSERT` logic.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "ETL Last Extract Time (YYYY-MM-DD HH:mm:ss)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59", "ETL Current Extract Time (YYYY-MM-DD HH:mm:ss)")

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

# SCEN_TASK_NO in {10}

In [ ]:
%sql
SELECT 1 AS task_10_status;

# SCEN_TASK_NO in {20}

In [ ]:
%sql
SELECT 1 AS task_20_status;

# SCEN_TASK_NO in {30}: Insert into Target Table

In [ ]:
%sql
INSERT INTO workspace.hr.trg_emp
  (
    employee_id,
    first_name,
    last_name,
    email,
    phone_number,
    hire_date,
    job_id,
    salary,
    commission_pct,
    manager_id,
    department_id
  )
SELECT
  employees.employee_id,
  employees.first_name,
  employees.last_name,
  employees.email,
  employees.phone_number,
  employees.hire_date,
  employees.job_id,
  employees.salary,
  employees.commission_pct,
  employees.manager_id,
  employees.department_id
FROM
  workspace.hr.employees AS employees;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp FROM workspace.hr.trg_emp;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** Original Oracle schema `HR` has been converted to `workspace.hr`. Table names `TRG_EMP` and `EMPLOYEES` have been converted to lowercase `trg_emp` and `employees` respectively.
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable to Delta Lake and Spark SQL.
3.  **Data Types:** No `CREATE TABLE` statement was provided in the original snippet. It's assumed that `workspace.hr.trg_emp` already exists with appropriate Spark SQL data types (e.g., `BIGINT` for integer IDs, `STRING` for VARCHAR2, `TIMESTAMP` for DATE/TIMESTAMP, `DECIMAL` or `DOUBLE` for SALARY/COMMISSION_PCT).
4.  **Empty SCEN_TASK_NOs:** Tasks `{10}` and `{20}` were empty in the original ODI script. Placeholder `SELECT 1 AS task_XX_status;` statements have been added to reflect their presence in the sequence, though they perform no functional operation.
5.  **ETL Parameters:** Standard ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, etc.) have been added as Databricks widgets and a temporary view (`v_etl_parameters`) created, following the standard notebook template. These parameters were not explicitly used in the provided ODI SQL snippet but are included for completeness per instructions.